# 25b — Pre-Adam Report Revision Pack v2

This notebook pulls together the fixed caveat recode, party-transition diagnostics, SDP validation v2, and Yorkshire case-study evidence into a clean pre-sharing pack.

It prepares the files and notes that should inform the next report revision before anything is shared with Adam.

Outputs are written to:

```text
data/processed/pre_adam_revision_v2/
```

In [2]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_DIRS = [
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "party_transition_diagnostics_v1",
    PROCESSED_DIR / "sdp_validation_v2",
    PROCESSED_DIR / "yorkshire_case_study_v1",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR,
]
OUTPUT_DIR = PROCESSED_DIR / "pre_adam_revision_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_revision_v2


In [3]:
def find_file(filename, required=False):
    for folder in INPUT_DIRS + [Path("/mnt/data")]:
        p = folder / filename
        if p.exists(): return p
    for root in [PROCESSED_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches: return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required: raise FileNotFoundError(filename)
    return None


def load(filename, required=False):
    p = find_file(filename, required=required)
    if p is None:
        print("Missing optional:", filename)
        return None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df


def save(df, filename):
    p = OUTPUT_DIR / filename
    df.to_csv(p, index=False)
    print("Saved", filename, df.shape)
    return p

## 25b.1 Load revised evidence layers

In [4]:
caveat_summary = load("north_west_caveat_recode_summary_v2.csv", required=True)
high_conf = load("north_west_high_confidence_review_v2.csv", required=True)
medium_conf = load("north_west_medium_confidence_review_v2.csv", required=True)
serious = load("north_west_serious_caveat_manual_review_v2.csv", required=True)
main_review = load("north_west_reportable_main_review_v2.csv", required=True)

party_all = load("north_west_party_transition_diagnostics_all_v1.csv", required=False)
conservative_transition = load("north_west_conservative_transition_wards_v1.csv", required=False)
labour_breakthrough = load("north_west_labour_stronghold_breakthrough_wards_v1.csv", required=False)
reform_ind = load("north_west_reform_independent_disruption_wards_v1.csv", required=False)
party_council = load("party_transition_summary_by_council_v1.csv", required=False)

sdp_profile = load("sdp_campaign_wards_profile_v2.csv", required=False)
sdp_count = load("sdp_candidate_count_check_v2.csv", required=False)
sdp_by_tribe = load("sdp_performance_by_dominant_tribe_v2.csv", required=False)
sdp_by_party = load("sdp_performance_by_latest_top_party_v2.csv", required=False)
sdp_unmatched = load("sdp_unmatched_results_review_v2.csv", required=False)

yorkshire_summary = load("yorkshire_case_study_summary_v1.csv", required=False)
yorkshire_wards = load("yorkshire_sdp_case_study_wards_v1.csv", required=False)
yorkshire_by_tribe = load("yorkshire_sdp_performance_by_tribe_v1.csv", required=False)
middleton = load("middleton_park_case_study_profile_v1.csv", required=False)

Loaded north_west_caveat_recode_summary_v2.csv: (4, 6) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_caveat_recode_summary_v2.csv
Loaded north_west_high_confidence_review_v2.csv: (671, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_high_confidence_review_v2.csv
Loaded north_west_medium_confidence_review_v2.csv: (153, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_medium_confidence_review_v2.csv
Loaded north_west_serious_caveat_manual_review_v2.csv: (1, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_serious_caveat_manual_review_v2.csv
Loaded north_west_reportable_main_review_v2.csv: (824, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_reportable_main_review_v2.csv
Loaded north_west_party_transition_diagnostics_all_v1.csv: (825, 22) from c:

## 25b.2 Relabel party-transition processes

This section records the diagnostic labels clearly. The Conservative diagnostic is deliberately renamed as **Conservative Legacy / Right-Adjacent Transition Terrain**, because the current proxy mostly identifies post-Conservative or right-disrupted terrain rather than current Conservative-held wards.

In [5]:
process_notes = pd.DataFrame([
    {
        "process_label": "Conservative Legacy / Right-Adjacent Transition Terrain",
        "old_label": "Conservative Transition Wards",
        "interpretation": "Wards showing signs of Conservative legacy, Reform/Independent disruption, or right-adjacent transition. This is not proof of current Conservative-held opportunity; it is a diagnostic for ex-Conservative or potentially Red Tory-adjacent terrain.",
        "report_use": "Use as a diagnostic lane, not as a main target score component yet.",
    },
    {
        "process_label": "Labour Stronghold Breakthrough Terrain",
        "old_label": "Labour Stronghold Breakthrough Wards",
        "interpretation": "Labour-dominant wards with demographic fit and possible safe-seat complacency/apathy dynamics. This is the Middleton Park-style breakthrough hypothesis.",
        "report_use": "Use as a breakthrough/build evidence lane requiring local candidate validation.",
    },
    {
        "process_label": "Reform / Independent Disruption Terrain",
        "old_label": "Reform Independent Disruption Wards",
        "interpretation": "Wards where conventional party structure appears fragmented by Reform/UKIP/Brexit, Independents or other non-main-party forces.",
        "report_use": "Use as evidence of political openness and disrupted party alignment.",
    },
])
save(process_notes, "pre_adam_party_process_label_notes_v2.csv")
display(process_notes)

Saved pre_adam_party_process_label_notes_v2.csv (3, 4)


,process_label,old_label,interpretation,report_use
0,Conservative Legacy / Right-Adjacent Transitio...,Conservative Transition Wards,"Wards showing signs of Conservative legacy, Re...","Use as a diagnostic lane, not as a main target..."
1,Labour Stronghold Breakthrough Terrain,Labour Stronghold Breakthrough Wards,Labour-dominant wards with demographic fit and...,Use as a breakthrough/build evidence lane requ...
2,Reform / Independent Disruption Terrain,Reform Independent Disruption Wards,Wards where conventional party structure appea...,Use as evidence of political openness and disr...


## 25b.3 Create headline metrics for the revised report

In [6]:
metrics = {
    "run_date": datetime.now().isoformat(timespec="seconds"),
    "high_confidence_rows": len(high_conf),
    "medium_confidence_rows": len(medium_conf),
    "serious_caveat_rows": len(serious),
    "main_report_rows": len(main_review),
}
if sdp_profile is not None:
    metrics["sdp_rows_total"] = len(sdp_profile)
    metrics["sdp_rows_matched_to_model"] = int(sdp_profile.get("matched_to_model_v2", pd.Series(False)).fillna(False).astype(bool).sum()) if "matched_to_model_v2" in sdp_profile.columns else int(sdp_profile.get("initial_watchlist_score", pd.Series()).notna().sum())
    metrics["sdp_max_vote_share"] = float(sdp_profile.get("sdp_vote_share_effective", pd.Series(dtype=float)).max())
if yorkshire_summary is not None and len(yorkshire_summary):
    for c in yorkshire_summary.columns:
        metrics[f"yorkshire_{c}"] = yorkshire_summary.iloc[0][c]

headline = pd.DataFrame([metrics])
save(headline, "pre_adam_headline_metrics_v2.csv")
display(headline.T)

Saved pre_adam_headline_metrics_v2.csv (1, 15)


,0
run_date,2026-05-28T14:54:34
high_confidence_rows,671
medium_confidence_rows,153
serious_caveat_rows,1
main_report_rows,824
sdp_rows_total,11
sdp_rows_matched_to_model,7
sdp_max_vote_share,NaN
yorkshire_case_study_rows,100
yorkshire_matched_to_model_rows,100


## 25b.4 Prepare report-ready extracts

In [7]:
# Confidence extracts.
save(high_conf.sort_values("initial_watchlist_score", ascending=False).head(100), "pre_adam_high_confidence_top100_v2.csv")
save(medium_conf.sort_values("initial_watchlist_score", ascending=False).head(100), "pre_adam_medium_confidence_top100_v2.csv")
save(serious.sort_values("initial_watchlist_score", ascending=False).head(100), "pre_adam_serious_caveat_top100_v2.csv")

# Party process extracts with corrected naming.
if conservative_transition is not None:
    ct = conservative_transition.copy()
    ct["process_label_v2"] = "Conservative Legacy / Right-Adjacent Transition Terrain"
    save(ct.head(100), "pre_adam_conservative_legacy_right_adjacent_transition_top100_v2.csv")
if labour_breakthrough is not None:
    lb = labour_breakthrough.copy()
    lb["process_label_v2"] = "Labour Stronghold Breakthrough Terrain"
    save(lb.head(100), "pre_adam_labour_stronghold_breakthrough_top100_v2.csv")
if reform_ind is not None:
    ri = reform_ind.copy()
    ri["process_label_v2"] = "Reform / Independent Disruption Terrain"
    save(ri.head(100), "pre_adam_reform_independent_disruption_top100_v2.csv")
if party_council is not None:
    save(party_council, "pre_adam_party_transition_summary_by_council_v2.csv")

# SDP validation extracts.
if sdp_count is not None:
    save(sdp_count, "pre_adam_sdp_candidate_count_check_v2.csv")
if sdp_by_tribe is not None:
    save(sdp_by_tribe, "pre_adam_sdp_performance_by_tribe_v2.csv")
if sdp_by_party is not None:
    save(sdp_by_party, "pre_adam_sdp_performance_by_latest_party_v2.csv")
if sdp_profile is not None:
    save(sdp_profile.sort_values("sdp_vote_share_effective", ascending=False).head(100), "pre_adam_sdp_highest_vote_share_cases_v2.csv")
if sdp_unmatched is not None:
    save(sdp_unmatched, "pre_adam_sdp_unmatched_rows_for_review_v2.csv")

# Yorkshire extracts.
if yorkshire_wards is not None:
    save(yorkshire_wards, "pre_adam_yorkshire_sdp_case_study_wards_v2.csv")
if yorkshire_by_tribe is not None:
    save(yorkshire_by_tribe, "pre_adam_yorkshire_sdp_performance_by_tribe_v2.csv")
if middleton is not None:
    save(middleton, "pre_adam_middleton_park_case_study_profile_v2.csv")

Saved pre_adam_high_confidence_top100_v2.csv (100, 74)
Saved pre_adam_medium_confidence_top100_v2.csv (100, 74)
Saved pre_adam_serious_caveat_top100_v2.csv (1, 74)
Saved pre_adam_conservative_legacy_right_adjacent_transition_top100_v2.csv (100, 23)
Saved pre_adam_labour_stronghold_breakthrough_top100_v2.csv (100, 23)
Saved pre_adam_reform_independent_disruption_top100_v2.csv (100, 23)
Saved pre_adam_party_transition_summary_by_council_v2.csv (35, 8)
Saved pre_adam_sdp_candidate_count_check_v2.csv (6, 4)
Saved pre_adam_sdp_performance_by_tribe_v2.csv (5, 8)
Saved pre_adam_sdp_performance_by_latest_party_v2.csv (6, 8)
Saved pre_adam_sdp_highest_vote_share_cases_v2.csv (11, 57)
Saved pre_adam_sdp_unmatched_rows_for_review_v2.csv (4, 57)
Saved pre_adam_yorkshire_sdp_case_study_wards_v2.csv (53, 15)
Saved pre_adam_yorkshire_sdp_performance_by_tribe_v2.csv (6, 9)
Saved pre_adam_middleton_park_case_study_profile_v2.csv (4, 72)


## 25b.5 Draft report revision notes

In [8]:
notes = f"""
# Pre-Adam Report Revision Notes v2

## Current status

The North West structural model should now be presented as a **breakthrough and organisational build model**, not a council-control model and not a final target-seat list.

## Caveat treatment

The caveat language has been revised from a blunt clean/caveated split into confidence bands:

- High confidence: usable ward-level evidence, no major caveat.
- Medium confidence: usable evidence with boundary, county-derived or proxy caveat.
- Serious caveat / manual review: missing or invalid core electoral evidence.

This avoids showing Adam technical mapping issues as if they were strategic warnings.

## Party process labels

Use the following labels:

1. Conservative Legacy / Right-Adjacent Transition Terrain  
   This captures ex-Conservative, Reform/Independent-disrupted or right-adjacent terrain. It is not proof that current Conservative-held wards are being identified.

2. Labour Stronghold Breakthrough Terrain  
   This captures the Middleton Park-style hypothesis: Labour-dominant wards where safe-seat complacency, low turnout and demographic fit may create breakthrough conditions.

3. Reform / Independent Disruption Terrain  
   This captures wards where the established party structure is already fractured.

## SDP validation

SDP campaign evidence should be treated as the first true validation layer. The current SDP validation notebook calculates effective vote share and improves ward matching, but 2026 data remains provisional and should be added using mapping-confidence fields.

## Yorkshire case study

The Yorkshire case study should be used to test which tribes and ward types have already produced SDP traction, especially Middleton Park, Dearne South, Wath, Doncaster/Barnsley/Rotherham/Sheffield and wider Leeds evidence.

## Report framing

The revised report should use the following three-way distinction:

1. Breakthrough geography — where near-term ward breakthrough appears plausible.
2. Build geography — where demographic fit exists but electoral evidence is weaker.
3. Control geography — not evidenced by Model v1; requires a separate seat-simulation model.

The report should explicitly state that V1 does not forecast council control, vote share or seat totals.
"""

path = OUTPUT_DIR / "pre_adam_report_revision_notes_v2.md"
path.write_text(notes, encoding="utf-8")
print(path)
print(notes)

c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_revision_v2\pre_adam_report_revision_notes_v2.md

# Pre-Adam Report Revision Notes v2

## Current status

The North West structural model should now be presented as a **breakthrough and organisational build model**, not a council-control model and not a final target-seat list.

## Caveat treatment

The caveat language has been revised from a blunt clean/caveated split into confidence bands:

- High confidence: usable ward-level evidence, no major caveat.
- Medium confidence: usable evidence with boundary, county-derived or proxy caveat.
- Serious caveat / manual review: missing or invalid core electoral evidence.

This avoids showing Adam technical mapping issues as if they were strategic warnings.

## Party process labels

Use the following labels:

1. Conservative Legacy / Right-Adjacent Transition Terrain  
   This captures ex-Conservative, Reform/Independent-disrupted or right-adjacent terrain. It is not proof that c

## 25b.6 Manifest

In [9]:
manifest = []
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    manifest.append({"file": p.name, "path": str(p), "type": "csv"})
for p in sorted(OUTPUT_DIR.glob("*.md")):
    manifest.append({"file": p.name, "path": str(p), "type": "markdown"})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUTPUT_DIR / "pre_adam_revision_manifest_v2.csv", index=False)
display(manifest_df)

,file,path,type
0,pre_adam_conservative_legacy_right_adjacent_tr...,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
1,pre_adam_headline_metrics_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
2,pre_adam_high_confidence_top100_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
3,pre_adam_labour_stronghold_breakthrough_top100...,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
4,pre_adam_medium_confidence_top100_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
5,pre_adam_middleton_park_case_study_profile_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
6,pre_adam_party_process_label_notes_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
7,pre_adam_party_transition_summary_by_council_v...,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
8,pre_adam_reform_independent_disruption_top100_...,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
9,pre_adam_sdp_candidate_count_check_v2.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv
